# Producción - Uso del modelo entrenado

In [51]:
# Importar librerías necesarias para cargar el modelo y procesar datos
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tensorflow.keras.models import load_model # función de Keras que reconstruye un modelo guardado en .keras
from pathlib import Path

## Cargar el modelo entrenado

In [52]:
# Ruta al archivo .keras
MODEL_PATH = Path("../models/bankruptcy_smote_model.keras")

model = load_model(MODEL_PATH)

print(f"Modelo cargado desde: {MODEL_PATH}")
print(f"Tipo: {type(model).__name__}")

Modelo cargado desde: ..\models\bankruptcy_smote_model.keras
Tipo: Sequential


In [53]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_3 (Dense)                 │ (None, 30)             │           960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 1)              │            31 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,975 (11.62 KB)

 Trainable params: 991 (3.87 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 1,984 (7.75 KB)

In [54]:
# Cargar el CSV con la empresa de prueba (datos crudos)
empresa_df = pd.read_csv("../production-data/empresa_no_quiebra.csv")

TARGET = "Bankrupt?"

# Separamos el target solo para poder comparar después contra la predicción
y_real = empresa_df[TARGET].values[0]

# Quitamos también la columna constante Net Income Flag que no aporta información
col_constante = [c for c in empresa_df.columns if c.strip() == 'Net Income Flag'][0]
X_empresa = empresa_df.drop(columns=[TARGET, col_constante]) # 94 features financieras crudas

print(f"Empresa cargada con {X_empresa.shape[1]} features crudas")
print(f"Valor real conocido: Bankrupt? = {y_real} ({'quiebra' if y_real == 1 else 'no quiebra'})")

Empresa cargada con 94 features crudas
Valor real conocido: Bankrupt? = 0 (no quiebra)


In [55]:
# Cargar los transformadores ya ajustados durante el preprocesamiento
import joblib

scaler = joblib.load("../split-dataset/preprocessed/scaler.pkl") # QuantileTransformer ajustado con el train original
pca = joblib.load("../split-dataset/preprocessed/pca.pkl") # PCA ajustado con el train original

# Aplicar las MISMAS transformaciones que se aplicaron en entrenamiento
X_scaled = scaler.transform(X_empresa) # escala las 94 features a distribución normal estándar
X_pca = pca.transform(X_scaled) # proyecta a los 31 componentes principales aprendidos del train

print(f"Shape después de scaler: {X_scaled.shape}")
print(f"Shape después de PCA:    {X_pca.shape} (31 dimensiones)")

Shape después de scaler: (1, 94)
Shape después de PCA:    (1, 31) (31 dimensiones)


c:\Users\angel\Desktop\ai-bankruptcy-prediction\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but PCA was fitted with feature names
  warnings.warn(


In [ ]:
# Predicción del modelo sobre los datos transformados
y_proba = model.predict(X_pca, verbose=0).ravel()[0] # probabilidad de quiebra (entre 0 y 1) que devuelve la sigmoid
y_pred = int(y_proba >= 0.5) # convertir a etiqueta binaria con threshold de 0.5 (>=0.5 → 1, <0.5 → 0)

print(f"Probabilidad de quiebra predicha: {y_proba:.4f}")
print(f"Predicción del modelo:            Bankrupt? = {y_pred} ({'quiebra' if y_pred == 1 else 'no quiebra'})")
print(f"Valor real conocido:              Bankrupt? = {y_real} ({'quiebra' if y_real == 1 else 'no quiebra'})")
print(f"\n{'Predicción correcta' if y_pred == y_real else 'Predicción incorrecta'}")

Probabilidad de quiebra predicha: 0.0086
Predicción del modelo:            Bankrupt? = 0 (no quiebra)
Valor real conocido:              Bankrupt? = 0 (no quiebra)

Predicción correcta ✓
